# Libraries


In [1]:
import os
import requests
import pandas as pd, json, ast
import requests
import json
from datetime import datetime, timedelta
import time
import ast

import paho.mqtt.client as mqtt

# 1. DATA RETRIEVAL

## 1.1. Declaring Constants 

In [ ]:
# ADD API KEY HERE
API_KEY="" # API Key add here

In [3]:
#Configs
BASE_URL = "https://api.openelectricity.org.au/v4"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
INTERVAL = "5m"

# Date range for 1st week of October 2025, End date is not inclusive.
START_DATE = datetime(2025, 10, 1)
END_DATE = datetime(2025, 10, 8)
REQUEST_DELAY = 0  # seconds between requests to avoid API limits

## 1.2. Fetching the list of Facilities

Fetch the list of all facilities for a given electricity network (default=NEM),  
Returns: pd.DataFrame: Facility metadata including name, region, code, location, and units.

In [4]:
def fetch_all_facilities(network: str = "NEM") -> pd.DataFrame:
    
    url = f"{BASE_URL}/facilities/"
    params = {
        "network_code": "NEM",
    }

    print("IN PROGRESS Fetching all facilities...")
    r = requests.get(url, headers=HEADERS, params=params)
    if r.status_code >= 400:
        print("API Error:")
        try:
            print(r.json())
        except Exception:
            print(r.text)
        r.raise_for_status()

    data = r.json().get("data", [])
    rows = []
    for entry in data:
        rows.append({
            "name": entry.get("name"),
      "network_id": entry.get("network_id"),
      "network_region": entry.get("network_region"),
      "description": entry.get("description"),
      "location": entry.get("location"),
       "code": entry.get("code"),

            "units": entry.get("units"),
        
          "created_at": entry.get("created_at"),
          "updated_at": entry.get("updated_at"),
       
       
        })

    df = pd.DataFrame(rows)
    print(f"SUCCESS Retrieved {len(df)} facilities")
    return df

facilities_df = fetch_all_facilities()


IN PROGRESS Fetching all facilities...
SUCCESS Retrieved 625 facilities


### 1.2.1. PER-FACILITY POWER GENERATED (5 min interval)

fetches 5-minute interval power generation data for every facility in the NEM network.  

In [5]:
#fetches 5-minute interval power generation data for every facility in the NEM network.  
def fetch_facility_power(facility_id: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Fetch 5-minute power data for a single facility and flatten it."""
    url = f"{BASE_URL}/data/facilities/NEM"
    #setting request parameters 
    params = {
        "metrics": ["power"],
        "interval": INTERVAL,
        "facility_code": facility_id,
        "date_start": START_DATE,
        "date_end": END_DATE,
        
    }

    # Make request and handle API / network issues safely
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed to fetch power for {facility_id}: {e}")
        return pd.DataFrame()

    # Check if the API returned success
    if not json_data.get("success", False):
        print(f" API returned failure for {facility_id}: {json_data.get('error')}")
        return pd.DataFrame()

    data = json_data.get("data", [])
    rows = []

    # Flatten nested JSON:
    for entry in data:
        unit_code = entry.get("unit", "MW")
        for res in entry.get("results", []):
            for point in res.get("data", []):
                timestamp, value = point
                rows.append({
                    "facility_id": facility_id,
                    "unit_code": unit_code,
                    "timestamp": timestamp,
                    "power_MW": value
                })

    return pd.DataFrame(rows)



# Step 1: Load facilities CSV (if needed)
facility_ids = facilities_df["code"]
print(f"IN PROGRESS Fetching power for {len(facility_ids)} facilities...")

all_power = []
count =1
# Step 2: Iterate over facilities
for facility_id in facility_ids:
    print(f"Fetching power for facility {facility_id}...count no "+ str(count))
    count+=1
    power_df = fetch_facility_power(facility_id, START_DATE, END_DATE)
    if power_df.empty:
        print(f" No data for {facility_id}")
    all_power.append(power_df)
    time.sleep(REQUEST_DELAY)

# Step 3: Combine all facilities
power_df_raw = pd.concat(all_power, ignore_index=True)


IN PROGRESS Fetching power for 625 facilities...
Fetching power for facility ADP...count no 1
Fetching power for facility ALBANY...count no 2
 Failed to fetch power for ALBANY: 404 Client Error: Not Found for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=ALBANY&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
 No data for ALBANY
Fetching power for facility ALDGASF...count no 3
Fetching power for facility AMCORGR...count no 4
 Failed to fetch power for AMCORGR: 404 Client Error: Not Found for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=AMCORGR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
 No data for AMCORGR
Fetching power for facility ANGASTON...count no 5
Fetching power for facility APS...count no 6
 Failed to fetch power for APS: 404 Client Error: Not Found for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metri

### 1.2.2. PER-FACILITY CO2 EMISSIONS (5 min interval)


fetches 5-minute interval emnission data for every facility in the NEM network.  

In [6]:
#Fetches 5-minute emissions data for a single facility and returns a flattened DataFrame.
def fetch_facility_emissions(facility_id: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Fetch 5-minute emissions data for a single facility and flatten it."""
    url = f"{BASE_URL}/data/facilities/NEM"
    # Passing Parameters
    params = {
        "metrics": ["emissions"],
        "interval": INTERVAL,
        "facility_code": facility_id,
        "date_start": START_DATE,
        "date_end": END_DATE,
    }

    # Make request and safely handle connection / API issues
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed to fetch emissions for {facility_id}: {e}")
        return pd.DataFrame()

    # Check API success flag
    if not json_data.get("success", False):
        print(f" API returned failure for {facility_id}: {json_data.get('error')}")
        return pd.DataFrame()

    data = json_data.get("data", [])
    rows = []

    # Flatten nested structure
    for entry in data:
        unit_code = entry.get("unit", "tCO2e")  # adjusted unit label
        for res in entry.get("results", []):
            for point in res.get("data", []):
                timestamp, value = point
                rows.append({
                    "facility_id": facility_id,
                    "unit_code": unit_code,
                    "timestamp": timestamp,
                    "emission": value
                })

    return pd.DataFrame(rows)


print(f"IN PROGRESS Fetching emissions for {len(facility_ids)} facilities...")

all_emissions = []

# Step 2: Iterate over facilities
count = 1
for facility_id in facility_ids:
    print(f"Fetching emissions for facility {facility_id}... count no {count}")
    count += 1
    emissions_df = fetch_facility_emissions(facility_id, START_DATE, END_DATE)
    if emissions_df.empty:
        print(f" No data for {facility_id}")
    all_emissions.append(emissions_df)
    time.sleep(REQUEST_DELAY)

# Step 3: Combine all facilities
emissions_df_raw = pd.concat(all_emissions, ignore_index=True)



IN PROGRESS Fetching emissions for 625 facilities...
Fetching emissions for facility ADP... count no 1
Fetching emissions for facility ALBANY... count no 2
 Failed to fetch emissions for ALBANY: 404 Client Error: Not Found for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=ALBANY&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
 No data for ALBANY
Fetching emissions for facility ALDGASF... count no 3
Fetching emissions for facility AMCORGR... count no 4
 Failed to fetch emissions for AMCORGR: 404 Client Error: Not Found for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=AMCORGR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
 No data for AMCORGR
Fetching emissions for facility ANGASTON... count no 5
Fetching emissions for facility APS... count no 6
 Failed to fetch emissions for APS: 404 Client Error: Not Found for url: https://a

### 1.2.3. PER-MARKET POWER PRICE AND DEMAND (5 min interval)


fetches 5-minute market data (spot price and demand) for each network region of the NEM.

In [7]:
NETWORK = "NEM"
NETWORK_REGIONS = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"]

# SETTINGS
NETWORK_REGIONS = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"]
METRICS = ["price", "demand"]

def fetch_market_metric(network: str, region: str, metric: str, start: datetime, end: datetime) -> pd.DataFrame:
    
    url = f"{BASE_URL}/market/network/{network}"
    params = {
        "metrics": metric,
        "interval": INTERVAL,
        "network_region": region,
        "date_start": START_DATE,
        "date_end": END_DATE,
    }

    # Request + error handling
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed {metric.upper()} {region} {start.date()}: {e}")
        return pd.DataFrame()

    # API error response handling
    if not json_data.get("success", False):
        print(f" API returned failure for {metric.upper()} {region}: {json_data.get('error')}")
        return pd.DataFrame()

    # Flatten nested JSON
    rows = []
    for entry in json_data.get("data", []):
        unit = entry.get("unit", "")
        for res in entry.get("results", []):
            for point in res.get("data", []):
                if len(point) == 2:
                    timestamp, value = point
                    rows.append({
                        "timestamp": timestamp,
                        "network": network,
                        "network_region": region,
                        f"value_{metric}": value,
                        f"unit_{metric}": unit,
                    })

    df = pd.DataFrame(rows)
    print(f"SUCCESSFUL {region}: {metric.upper()} {start.date()} → {len(df)} rows")
    return df



# Collect one day at a time to avoid overwhelming API
all_data = []
current_date = START_DATE

while current_date < END_DATE:
    next_date = current_date + timedelta(days=1)

    for region in NETWORK_REGIONS:
        region_data = []
        for metric in METRICS:
            df = fetch_market_metric(NETWORK, region, metric, current_date, next_date)
            if not df.empty:
                region_data.append(df)

        # Merge price + demand for this region/day
        if region_data:
            merged_df = region_data[0]
            for df in region_data[1:]:
                merged_df = pd.merge(
                    merged_df,
                    df,
                    on=["timestamp", "network", "network_region"],
                    how="outer"
                )
            all_data.append(merged_df)

    current_date = next_date
    
# Final concatenation of all data
if not all_data:
    print(" No market data retrieved for any region or day.")
 
market_raw = pd.concat(all_data, ignore_index=True)


SUCCESSFUL NSW1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL QLD1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL QLD1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL SA1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL SA1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL TAS1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL TAS1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL VIC1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL VIC1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL NSW1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL QLD1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL QLD1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL SA1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL SA1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL TAS1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL TAS1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL VIC1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL VIC1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL NSW1: PRICE 2025-10-03 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025

## 1.3. RAW dataframes list

In [8]:
facilities_df

,name,network_id,network_region,description,location,code,units,created_at,updated_at
0,Adelaide Desalination,NEM,SA1,"<p>The Adelaide Desalination plant (ADP), form...","{'lat': -35.096948, 'lng': 138.484061}",ADP,"[{'code': 'ADPBA1G', 'code_display': '0ADPBA1G...",2023-10-18T04:34:30Z,2026-06-23T03:48:08Z
1,Albany,WEM,WEM,<p>Albany wind and Grasmere farms are two wind...,"{'lat': -35.055759, 'lng': 117.777048}",ALBANY,"[{'code': 'ALBANY_WF1', 'code_display': 'ALBAN...",2023-10-18T04:34:30Z,2026-08-14T00:15:29Z
2,Aldoga,NEM,QLD1,<p>The Aldoga Solar Farm will be approximately...,"{'lat': -23.815177, 'lng': 151.05811}",ALDGASF,"[{'code': 'ALDGASF1', 'code_display': 'ALDGASF...",2025-01-31T04:19:33Z,2026-08-14T00:15:29Z
3,Amcor Glass,NEM,SA1,<p></p>,"{'lat': -34.847716, 'lng': 138.581503}",AMCORGR,"[{'code': 'AMCORGR', 'code_display': 'AMCORGR'...",2023-10-18T04:34:32Z,2026-08-14T00:15:44Z
4,Angaston,NEM,SA1,<p>Angaston Power Station is a diesel-powered ...,"{'lat': -34.503389, 'lng': 139.02458}",ANGASTON,"[{'code': 'ANGAST1', 'code_display': 'ANGAST1'...",2023-10-18T04:34:32Z,2026-06-16T04:42:05Z
...,...,...,...,...,...,...,...,...,...
620,Yarrawonga,NEM,VIC1,<p>Yarrawonga Power Station is adjacent to the...,"{'lat': -36.009466, 'lng': 145.999568}",YWNGAHYD,"[{'code': 'YWNGAHYD', 'code_display': 'YWNGAHY...",2023-10-18T04:35:33Z,2026-03-20T03:55:57Z
621,Yarwun,NEM,QLD1,<p></p>,"{'lat': -23.8309, 'lng': 151.151136}",YARWUN,"[{'code': 'YARWUN_1', 'code_display': 'YARWUN_...",2023-10-18T04:35:33Z,2026-03-17T23:52:47Z
622,Yatpool,NEM,VIC1,<p>The 258-hectare Yatpool Solar Farm in north...,"{'lat': -34.396363, 'lng': 142.173809}",YATSF1,"[{'code': 'YATSF1', 'code_display': 'YATSF1', ...",2023-10-18T04:35:33Z,2026-03-19T03:44:32Z
623,Yawong,NEM,VIC1,<p>The Yawong Wind Farm was developed and cons...,"{'lat': -36.481161, 'lng': 143.366605}",YAWWF,"[{'code': 'YAWWF1', 'code_display': 'YAWWF1', ...",2023-10-18T04:35:33Z,2026-08-14T00:15:42Z


In [9]:
power_df_raw 

,facility_id,unit_code,timestamp,power_MW
0,ADP,MW,2025-10-01T00:00:00+10:00,-0.004
1,ADP,MW,2025-10-01T00:05:00+10:00,-0.046
2,ADP,MW,2025-10-01T00:10:00+10:00,0.000
3,ADP,MW,2025-10-01T00:15:00+10:00,0.003
4,ADP,MW,2025-10-01T00:20:00+10:00,-0.018
...,...,...,...,...
1084360,YENDONWF,MW,2025-10-07T23:35:00+10:00,88.630
1084361,YENDONWF,MW,2025-10-07T23:40:00+10:00,91.890
1084362,YENDONWF,MW,2025-10-07T23:45:00+10:00,89.030
1084363,YENDONWF,MW,2025-10-07T23:50:00+10:00,85.830


In [10]:
emissions_df_raw 

,facility_id,unit_code,timestamp,emission
0,ADP,t,2025-10-01T00:00:00+10:00,0.0
1,ADP,t,2025-10-01T00:05:00+10:00,0.0
2,ADP,t,2025-10-01T00:10:00+10:00,0.0
3,ADP,t,2025-10-01T00:15:00+10:00,0.0
4,ADP,t,2025-10-01T00:20:00+10:00,0.0
...,...,...,...,...
1084360,YENDONWF,t,2025-10-07T23:35:00+10:00,0.0
1084361,YENDONWF,t,2025-10-07T23:40:00+10:00,0.0
1084362,YENDONWF,t,2025-10-07T23:45:00+10:00,0.0
1084363,YENDONWF,t,2025-10-07T23:50:00+10:00,0.0


In [11]:
market_raw

,timestamp,network,network_region,value_price,unit_price,value_demand,unit_demand
0,2025-10-01T00:00:00+10:00,NEM,NSW1,56.98,$/MWh,7105.57,MW
1,2025-10-01T00:05:00+10:00,NEM,NSW1,80.01,$/MWh,7170.68,MW
2,2025-10-01T00:10:00+10:00,NEM,NSW1,58.40,$/MWh,7158.50,MW
3,2025-10-01T00:15:00+10:00,NEM,NSW1,80.01,$/MWh,7177.80,MW
4,2025-10-01T00:20:00+10:00,NEM,NSW1,65.00,$/MWh,7085.54,MW
...,...,...,...,...,...,...,...
70555,2025-10-07T23:35:00+10:00,NEM,VIC1,104.22,$/MWh,4709.39,MW
70556,2025-10-07T23:40:00+10:00,NEM,VIC1,103.68,$/MWh,4696.86,MW
70557,2025-10-07T23:45:00+10:00,NEM,VIC1,95.86,$/MWh,4639.93,MW
70558,2025-10-07T23:50:00+10:00,NEM,VIC1,95.88,$/MWh,4616.01,MW


# 2. DATA INTEGRATION/CLEANING

## 2.1. Cleaning and Combining Power-Emission

Merge facility power and emissions data on facility ID and timestamp. Outer join keeps rows even if power or emissions is missing for a time period

In [12]:
power_emission_raw_stage_1 = pd.merge(
    power_df_raw,
    emissions_df_raw,
    on=['facility_id', 'timestamp'],
    how='outer' 
).sort_values(by=['timestamp','facility_id']).reset_index(drop=True)
print(power_emission_raw_stage_1)

        facility_id unit_code_x                  timestamp  power_MW  \
0               ADP          MW  2025-10-01T00:00:00+10:00    -0.004   
1               ADP          MW  2025-10-01T00:00:00+10:00    -0.004   
2               ADP          MW  2025-10-01T00:00:00+10:00    -0.004   
3               ADP          MW  2025-10-01T00:00:00+10:00     0.000   
4               ADP          MW  2025-10-01T00:00:00+10:00     0.000   
...             ...         ...                        ...       ...   
2425586     YARANSF          MW  2025-10-07T23:55:00+10:00     0.000   
2425587      YARWUN          MW  2025-10-07T23:55:00+10:00   138.990   
2425588      YATSF1          MW  2025-10-07T23:55:00+10:00     0.000   
2425589    YENDONWF          MW  2025-10-07T23:55:00+10:00    89.870   
2425590        YSWF          MW  2025-10-07T23:55:00+10:00    11.600   

        unit_code_y  emission  
0                 t    0.0000  
1                 t    0.0000  
2                 t    0.0000  
3      

Here we :
- Take absolute value of Power since some Power is in -ve which cannot be true for a power plant

- drop redundant columns
- drop rows with 0 in both power and emission

In [13]:
power_emission_raw_stage_1['power_in_MW'] = power_emission_raw_stage_1['power_MW'].abs()
power_emission_raw_stage_1 = power_emission_raw_stage_1[~((power_emission_raw_stage_1['power_MW'] == 0) & (power_emission_raw_stage_1['emission'] == 0))]
power_emission_raw_stage_1 = power_emission_raw_stage_1.drop(columns=[col for col in ['unit_code_x', 'unit_code_y','power_MW'] if col in power_emission_raw_stage_1.columns])


print(power_emission_raw_stage_1.reset_index(drop=True))

        facility_id                  timestamp  emission  power_in_MW
0               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
1               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
2               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
3               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
4               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
...             ...                        ...       ...          ...
1586412    YALLOURN  2025-10-07T23:55:00+10:00   38.9360        0.000
1586413      YAMBUK  2025-10-07T23:55:00+10:00    0.0000       14.600
1586414      YARWUN  2025-10-07T23:55:00+10:00    7.1011      138.990
1586415    YENDONWF  2025-10-07T23:55:00+10:00    0.0000       89.870
1586416        YSWF  2025-10-07T23:55:00+10:00    0.0000       11.600

[1586417 rows x 4 columns]


In [14]:

# Round to 3 decimal places (numeric precision)
power_emission_raw_stage_1["power_in_MW"] = power_emission_raw_stage_1["power_in_MW"].round(3)
power_emission_raw_stage_1["emission"] = power_emission_raw_stage_1["emission"].round(3)

# Format to always show exactly 3 decimal places
power_emission_raw_stage_1["power_in_MW"] = power_emission_raw_stage_1["power_in_MW"].map(lambda x: f"{x:.3f}")
power_emission_raw_stage_1["emission"] = power_emission_raw_stage_1["emission"].map(lambda x: f"{x:.3f}")

# Remove duplicates where timestamp, power, and emission are the same
power_emission_raw_stage_2 = power_emission_raw_stage_1.drop_duplicates(
    subset=["timestamp", "power_in_MW", "emission"]
).reset_index(drop=True)

power_emission_raw_stage_2

,facility_id,timestamp,emission,power_in_MW
0,ADP,2025-10-01T00:00:00+10:00,0.000,0.004
1,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400
2,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081
3,B2PS,2025-10-01T00:00:00+10:00,0.000,0.081
4,B2PS,2025-10-01T00:00:00+10:00,0.004,0.000
...,...,...,...,...
771373,YALLOURN,2025-10-07T23:55:00+10:00,38.936,360.562
771374,YAMBUK,2025-10-07T23:55:00+10:00,0.000,14.600
771375,YARWUN,2025-10-07T23:55:00+10:00,7.101,138.990
771376,YENDONWF,2025-10-07T23:55:00+10:00,0.000,89.870


## 2.2. Cleaning Facilities

In [15]:
facilities_df

,name,network_id,network_region,description,location,code,units,created_at,updated_at
0,Adelaide Desalination,NEM,SA1,"<p>The Adelaide Desalination plant (ADP), form...","{'lat': -35.096948, 'lng': 138.484061}",ADP,"[{'code': 'ADPBA1G', 'code_display': '0ADPBA1G...",2023-10-18T04:34:30Z,2026-06-23T03:48:08Z
1,Albany,WEM,WEM,<p>Albany wind and Grasmere farms are two wind...,"{'lat': -35.055759, 'lng': 117.777048}",ALBANY,"[{'code': 'ALBANY_WF1', 'code_display': 'ALBAN...",2023-10-18T04:34:30Z,2026-08-14T00:15:29Z
2,Aldoga,NEM,QLD1,<p>The Aldoga Solar Farm will be approximately...,"{'lat': -23.815177, 'lng': 151.05811}",ALDGASF,"[{'code': 'ALDGASF1', 'code_display': 'ALDGASF...",2025-01-31T04:19:33Z,2026-08-14T00:15:29Z
3,Amcor Glass,NEM,SA1,<p></p>,"{'lat': -34.847716, 'lng': 138.581503}",AMCORGR,"[{'code': 'AMCORGR', 'code_display': 'AMCORGR'...",2023-10-18T04:34:32Z,2026-08-14T00:15:44Z
4,Angaston,NEM,SA1,<p>Angaston Power Station is a diesel-powered ...,"{'lat': -34.503389, 'lng': 139.02458}",ANGASTON,"[{'code': 'ANGAST1', 'code_display': 'ANGAST1'...",2023-10-18T04:34:32Z,2026-06-16T04:42:05Z
...,...,...,...,...,...,...,...,...,...
620,Yarrawonga,NEM,VIC1,<p>Yarrawonga Power Station is adjacent to the...,"{'lat': -36.009466, 'lng': 145.999568}",YWNGAHYD,"[{'code': 'YWNGAHYD', 'code_display': 'YWNGAHY...",2023-10-18T04:35:33Z,2026-03-20T03:55:57Z
621,Yarwun,NEM,QLD1,<p></p>,"{'lat': -23.8309, 'lng': 151.151136}",YARWUN,"[{'code': 'YARWUN_1', 'code_display': 'YARWUN_...",2023-10-18T04:35:33Z,2026-03-17T23:52:47Z
622,Yatpool,NEM,VIC1,<p>The 258-hectare Yatpool Solar Farm in north...,"{'lat': -34.396363, 'lng': 142.173809}",YATSF1,"[{'code': 'YATSF1', 'code_display': 'YATSF1', ...",2023-10-18T04:35:33Z,2026-03-19T03:44:32Z
623,Yawong,NEM,VIC1,<p>The Yawong Wind Farm was developed and cons...,"{'lat': -36.481161, 'lng': 143.366605}",YAWWF,"[{'code': 'YAWWF1', 'code_display': 'YAWWF1', ...",2023-10-18T04:35:33Z,2026-08-14T00:15:42Z


In [16]:
# Step 1: Drop unwanted columns 
facilities_df_stage_1 = facilities_df.drop(columns=['description', 'created_at', 'updated_at'], errors='ignore')

# Step 2: Split 'location' dict into latitude and longitude
# If 'location' is a string, convert to dict first
facilities_df_stage_1['location'] = facilities_df_stage_1['location'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
facilities_df_stage_1['latitude'] = facilities_df_stage_1['location'].apply(lambda x: x.get('lat') if isinstance(x, dict) else None)
facilities_df_stage_1['longitude'] = facilities_df_stage_1['location'].apply(lambda x: x.get('lng') if isinstance(x, dict) else None)
facilities_df_stage_1 = facilities_df_stage_1.drop(columns=['location'])

#  Clean 'network_region' (remove trailing '1') 
facilities_df_stage_1['network_region'] = facilities_df_stage_1['network_region'].str.replace(r'1$', '', regex=True)

# Step 4: Extract first 'fueltech_id' from 'units' list 
# Convert stringified lists to Python lists (if needed)
facilities_df_stage_1['units'] = facilities_df_stage_1['units'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
facilities_df_stage_1['fuel_source'] = facilities_df_stage_1['units'].apply(
    lambda x: x[0]['fueltech_id'] if isinstance(x, list) and len(x) > 0 else None
)

# Optional: Drop original 'units' column if no longer needed
#facilities_df_stage_1 = facilities_df_stage_1.drop(columns=['units'])

# Final cleaned DataFrame
print(facilities_df_stage_1)

                      name network_id network_region      code  \
0    Adelaide Desalination        NEM             SA       ADP   
1                   Albany        WEM            WEM    ALBANY   
2                   Aldoga        NEM            QLD   ALDGASF   
3              Amcor Glass        NEM             SA   AMCORGR   
4                 Angaston        NEM             SA  ANGASTON   
..                     ...        ...            ...       ...   
620             Yarrawonga        NEM            VIC  YWNGAHYD   
621                 Yarwun        NEM            QLD    YARWUN   
622                Yatpool        NEM            VIC    YATSF1   
623                 Yawong        NEM            VIC     YAWWF   
624                 Yendon        NEM            VIC  YENDONWF   

                                                 units   latitude   longitude  \
0    [{'code': 'ADPBA1G', 'code_display': '0ADPBA1G... -35.096948  138.484061   
1    [{'code': 'ALBANY_WF1', 'code_display': 

## 2.3. Combine facilities and power-emission

Left-Join facility attributes (name, region, location etc.) onto the power-emissions data

In [17]:
power_emission_raw_stage_3 = pd.merge(
    power_emission_raw_stage_2,
    facilities_df_stage_1,
    how='left',
    left_on='facility_id',
    right_on='code'
)

In [18]:
power_emission_raw_stage_3

,facility_id,timestamp,emission,power_in_MW,name,network_id,network_region,code,units,latitude,longitude,fuel_source
0,ADP,2025-10-01T00:00:00+10:00,0.000,0.004,Adelaide Desalination,NEM,SA,ADP,"[{'code': 'ADPBA1G', 'code_display': '0ADPBA1G...",-35.096948,138.484061,battery_discharging
1,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400,Ararat,NEM,VIC,ARWF,"[{'code': 'ARWF1', 'code_display': 'ARWF1', 'f...",-37.238714,143.079243,wind
2,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081,Braemar 2,NEM,QLD,B2PS,"[{'code': 'BRAEMAR6', 'code_display': 'BRAEMAR...",-27.112874,150.905442,gas_ocgt
3,B2PS,2025-10-01T00:00:00+10:00,0.000,0.081,Braemar 2,NEM,QLD,B2PS,"[{'code': 'BRAEMAR6', 'code_display': 'BRAEMAR...",-27.112874,150.905442,gas_ocgt
4,B2PS,2025-10-01T00:00:00+10:00,0.004,0.000,Braemar 2,NEM,QLD,B2PS,"[{'code': 'BRAEMAR6', 'code_display': 'BRAEMAR...",-27.112874,150.905442,gas_ocgt
...,...,...,...,...,...,...,...,...,...,...,...,...
771373,YALLOURN,2025-10-07T23:55:00+10:00,38.936,360.562,Yallourn W,NEM,VIC,YALLOURN,"[{'code': 'YWPS4', 'code_display': 'YWPS4', 'f...",-38.177589,146.342556,coal_brown
771374,YAMBUK,2025-10-07T23:55:00+10:00,0.000,14.600,Yambuk,NEM,VIC,YAMBUK,"[{'code': 'YAMBUKWF', 'code_display': 'YAMBUKW...",-38.310953,142.014788,wind
771375,YARWUN,2025-10-07T23:55:00+10:00,7.101,138.990,Yarwun,NEM,QLD,YARWUN,"[{'code': 'YARWUN_1', 'code_display': 'YARWUN_...",-23.830900,151.151136,gas_ccgt
771376,YENDONWF,2025-10-07T23:55:00+10:00,0.000,89.870,Yendon,NEM,VIC,YENDONWF,"[{'code': 'YENDWF1', 'code_display': 'YENDWF1'...",-37.630407,144.023136,wind


In [19]:
power_emission_raw_stage_4 = power_emission_raw_stage_3.drop(columns=["network_id", "code"])
print(power_emission_raw_stage_4)

       facility_id                  timestamp emission power_in_MW  \
0              ADP  2025-10-01T00:00:00+10:00    0.000       0.004   
1             ARWF  2025-10-01T00:00:00+10:00    0.000      35.400   
2             B2PS  2025-10-01T00:00:00+10:00    0.004       0.081   
3             B2PS  2025-10-01T00:00:00+10:00    0.000       0.081   
4             B2PS  2025-10-01T00:00:00+10:00    0.004       0.000   
...            ...                        ...      ...         ...   
771373    YALLOURN  2025-10-07T23:55:00+10:00   38.936     360.562   
771374      YAMBUK  2025-10-07T23:55:00+10:00    0.000      14.600   
771375      YARWUN  2025-10-07T23:55:00+10:00    7.101     138.990   
771376    YENDONWF  2025-10-07T23:55:00+10:00    0.000      89.870   
771377        YSWF  2025-10-07T23:55:00+10:00    0.000      11.600   

                         name network_region  \
0       Adelaide Desalination             SA   
1                      Ararat            VIC   
2              

For Final Cleanup, we drop duplicates

In [20]:

power_emission_final = power_emission_raw_stage_4.drop_duplicates(
    subset=['facility_id', 'timestamp'],
    keep='first'  # or 'last' if you want to keep the last occurrence
)

# Optional: reset index
power_emission_final.reset_index(drop=True, inplace=True)

## 2.4. Price and Demand Cleanup

Clean Up Per Market Price and Demand

In [21]:
market_raw_stage_1 =market_raw

In [22]:
# Remove the trailing '1' from region codes (e.g., 'NSW1' to 'NSW')
market_raw_stage_1['network_region'] = market_raw_stage_1['network_region'].str.replace(r'1$', '', regex=True)

In [23]:
# Remove duplicate timestamp rows to avoid double counting price/demand
market_raw_stage_1 = market_raw_stage_1.drop_duplicates(
    subset=["timestamp", "value_price", "value_demand"]
).reset_index(drop=True)


# Drop network column (not needed after region-level aggregation)
market_raw_stage_2 = market_raw_stage_1.drop(columns=["network"])

# Rename metric columns and drop unit columns since units are implied
market_raw_stage_2 = market_raw_stage_2.rename(
    columns={
        "value_price": "value_price_in_$/MWh",
        "value_demand": "value_demand_in_MW"
    }
).drop(columns=["unit_price", "unit_demand"])
# Force all price and demand values to positive (market data sometimes returns signed numbers)
market_raw_stage_2["value_price_in_$/MWh"] = market_raw_stage_2["value_price_in_$/MWh"].abs()
market_raw_stage_2["value_demand_in_MW"] = market_raw_stage_2["value_demand_in_MW"].abs()

In [24]:
# Sort market data chronologically and by region for consistent merging
market_raw_stage_3 = market_raw_stage_2.sort_values(
    by=["timestamp", "network_region"]
).reset_index(drop=True)

## 2.5. Merging Power and Market Datasets

In [25]:
# Merge facility time-series data with market price & demand based on timestamp and region
merged_df = pd.merge(
    power_emission_final,market_raw_stage_3
    [["timestamp", "network_region", "value_price_in_$/MWh", "value_demand_in_MW"]],
    on=["timestamp", "network_region"],
    how="left"
)

# Quick sanity check
print("Merged shape:", merged_df.shape)
merged_df.head(5)

Merged shape: (412486, 12)


,facility_id,timestamp,emission,power_in_MW,name,network_region,units,latitude,longitude,fuel_source,value_price_in_$/MWh,value_demand_in_MW
0,ADP,2025-10-01T00:00:00+10:00,0.000,0.004,Adelaide Desalination,SA,"[{'code': 'ADPBA1G', 'code_display': '0ADPBA1G...",-35.096948,138.484061,battery_discharging,8.11,1564.92
1,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400,Ararat,VIC,"[{'code': 'ARWF1', 'code_display': 'ARWF1', 'f...",-37.238714,143.079243,wind,8.95,4893.49
2,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081,Braemar 2,QLD,"[{'code': 'BRAEMAR6', 'code_display': 'BRAEMAR...",-27.112874,150.905442,gas_ocgt,54.82,5989.24
3,BANGOWF,2025-10-01T00:00:00+10:00,0.000,97.699,Bango,NSW,"[{'code': 'BANGOWF2', 'code_display': 'BANGOWF...",-34.767208,148.921499,wind,56.98,7105.57
4,BARCSF,2025-10-01T00:00:00+10:00,0.000,0.100,Barcaldine,QLD,"[{'code': 'BARCSF1', 'code_display': 'BARCSF1'...",-23.547776,145.318817,solar_utility,54.82,5989.24


In [26]:
# Save to a new CSV for use in Stage 3
merged_df.to_csv("power_emission.csv", index=False)

# 3. Data Publishing via MQTT

## 3.1. MQTT configs and setup

In [27]:
MQTT_BROKER = "test.mosquitto.org"
MQTT_PORT   = 1883
TOPIC_POWER = "470461201/A2"  # topic for pub./sub
PUBLISH_DELAY = 0.1         # seconds between messages  
RELOAD_DELAY  = 0          # seconds to wait between rounds when REPEAT=True
END_WAIT_SECS = 60           # wait before replay restart
REPEAT = True                # keep publishing in a loop

## 3.2. Transforming Facilities Unit Data

Parsing and Extracting Unit Codes from Facility Metadata

In [28]:
def parse_units_codes(units_cell):
    """Return ONLY the 'code' values from the 'units' JSON."""
    if units_cell is None or (isinstance(units_cell, float) and pd.isna(units_cell)):
        return []
    obj = units_cell
    # If stored as a string (JSON or Python literal), try to parse it
    if isinstance(units_cell, str):
        s = units_cell.strip()
        if not s:
            return []
        try:
            obj = json.loads(s) #JSON
        except Exception:
            try:
                obj = ast.literal_eval(s) #Python
            except Exception:
                return []
    # Ensure we are working with a list of dicts
    if isinstance(obj, dict):
        obj = [obj]
    if not isinstance(obj, (list, tuple)):
        return []
    return [str(d.get("code")) for d in obj if isinstance(d, dict) and d.get("code")]

CSV_PATH = "power_emission.csv"
power_df = pd.read_csv(CSV_PATH)

# Extract clean list of codes and a display-friendly string
power_df["unit_codes"] = power_df["units"].apply(parse_units_codes)
power_df["unit_codes_str"] = power_df["unit_codes"].apply(lambda xs: ", ".join(xs) if xs else "—")

display(power_df[["facility_id","name","unit_codes","unit_codes_str"]].head(5))
# Ensure timestamps are datetime objects
power_df["timestamp"] = pd.to_datetime(power_df["timestamp"])






,facility_id,name,unit_codes,unit_codes_str
0,ADP,Adelaide Desalination,"[ADPBA1G, ADPBA1L, ADPBA1, ADPPV1, ADPPV2, ADP...","ADPBA1G, ADPBA1L, ADPBA1, ADPPV1, ADPPV2, ADPP..."
1,ARWF,Ararat,[ARWF1],ARWF1
2,B2PS,Braemar 2,"[BRAEMAR6, BRAEMAR5, BRAEMAR7]","BRAEMAR6, BRAEMAR5, BRAEMAR7"
3,BANGOWF,Bango,"[BANGOWF2, BANGOWF1]","BANGOWF2, BANGOWF1"
4,BARCSF,Barcaldine,[BARCSF1],BARCSF1


Initializing MQTT Client and Publish Callback

In [29]:
client = mqtt.Client()
client.connect(MQTT_BROKER, MQTT_PORT, 60)
client.loop_start()

def on_publish(client, userdata, mid):
    print(f"MQTT message with mid={mid} successfully published.")

client.on_publish = on_publish

/var/folders/0x/p1c52y6d1z57g2cq5s9yfqk40000gn/T/ipykernel_65636/3426122455.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


## 3.3. Publisher Setup (5. Continuous Execution)

Publishing Time-Series Power and Emissions Data as MQTT Stream. Duplicaton Avoided and Continuous Execution Executed

In [30]:
def publish_stream(power_df, delay_between_messages=PUBLISH_DELAY, delay_between_timestamps=RELOAD_DELAY):
    all_timestamps = sorted(power_df["timestamp"].unique())

    for ts in all_timestamps:
        print(f"\nPublishing data for timestamp: {ts}")

        # Pulling and Sending Data
        subset_power = power_df[power_df["timestamp"] == ts]
        for _, row in subset_power.iterrows():
            # To improve latency, ignoring row if power and emission are 0
            power_val    = float(row.get("power_in_MW", 0) or 0)
            emission_val = float(row.get("emission",    0) or 0)
            if power_val == 0 and emission_val == 0:
                continue 
            
            # Message Data
            msg_power = {
                "facility_id": row["facility_id"],
                "name": row["name"],
                "timestamp": str(row["timestamp"]),
                "power_MW": power_val,
                "emissions_tCO2": emission_val,
                "network_region": row["network_region"],
                "fuel_source": row["fuel_source"],
                "latitude": float(row["latitude"]),
                "longitude": float(row["longitude"]),
                "value_price_in_$/MWh": float(row.get("value_price_in_$/MWh")),
                "value_demand_in_MW": float(row.get("value_demand_in_MW")),
                "unit_codes": row.get("unit_codes"),
                "unit_codes_str": row.get("unit_codes_str")
            }
            client.publish(TOPIC_POWER, json.dumps(msg_power))
            print(f" Power published: {msg_power['facility_id']} ({msg_power['power_MW']} MW)")
            #Delay between messages is 0.1s
            time.sleep(delay_between_messages)

        #Delay between timestamp batches is 0s, can be set by changing RELOAD_DELAY 
        print(f"Finished timestamp {ts}. Waiting {delay_between_timestamps}s before next batch...")
        time.sleep(delay_between_timestamps)
      
    #Continous Execution, if we run out of data, we wait for 60s and replay the dataset again
    if REPEAT:
        print(f"End of data reached. Waiting {END_WAIT_SECS}s before restarting...")
        time.sleep(END_WAIT_SECS)
        publish_stream(power_df)
        return


## 3.4. Running Publisher

Publishing the Data

In [31]:
try:
    publish_stream(power_df)
except KeyboardInterrupt:
    print("\n Streaming stopped by user.")
finally:
    client.loop_stop()
    client.disconnect()
    print(" MQTT client disconnected")



Publishing data for timestamp: 2025-10-01 00:00:00+10:00
 Power published: ADP (0.004 MW)
MQTT message with mid=1 successfully published.
 Power published: ARWF (35.4 MW)
MQTT message with mid=2 successfully published.
 Power published: B2PS (0.081 MW)
MQTT message with mid=3 successfully published.
 Power published: BANGOWF (97.699 MW)
MQTT message with mid=4 successfully published.
 Power published: BARCSF (0.1 MW)MQTT message with mid=5 successfully published.

 Power published: BASTYAN (57.1 MW)
MQTT message with mid=6 successfully published.
 Power published: BAYSW (558.395 MW)MQTT message with mid=7 successfully published.

 Power published: BBATTERY (0.015 MW)
MQTT message with mid=8 successfully published.
 Power published: BHB (2.194 MW)MQTT message with mid=9 successfully published.

 Power published: BHWF (7.764 MW)
MQTT message with mid=10 successfully published.
 Power published: BIALAWF (69.172 MW)
MQTT message with mid=11 successfully published.
 Power published: BLUEGS